# Practical: Molecular dynamics simulations of a small protein domain

By:  [ Camilo Aponte-Santamaría, MPI for Polymer Research, Mainz Germany](https://mptg-cbp.github.io/caas/)


**Acknowledgement:** This tutorial is based on the [<ins>tutorial written by Bert de Groot</ins>](http://www3.mpibpc.mpg.de/groups/de_groot/compbio1/p2/index.html).

##1) Introduction

In this practical we will carry out a molecular dynamics (MD) simulation of a biological macromolecule: a small protein.  

Molecular dynamics (MD) simulations and related computational tools have nowadays become essential tools for the study of biomolecular systems. Over the last decade, it has been successfully
elucidated key biological processes such as enzyme catalysis, ribosomal translocation, DNA repair, membrane permeation, protein folding, macromolecular crowding and viral assembly, among many
others. The use of this technique enables monitoring the dynamics of biomolecules in an unprecedentedly broad spatio-temporal range, from pico- to milli-seconds and from thousands to hundreds of millions of atoms. Moreover, recent advances in computing power, together with optimized algorithms, machine learning strategies, and new sampling methodologies, are pushing these boundaries further, while increasing the precision and accuracy. Simulations do not only help interpret experimental data but also complement experimental techniques such as Cryo-EM, FRET, SAXS, or NMR to reveal key structure, dynamic, and energetic information of biomolecules.

Today, we will simulate the dynamics of a small, typical protein domain: the B1 domain of protein G. B1 is one of the domains of protein G, a member of an important class of proteins, which form IgG binding receptors on the surface of certain Staphylococcal and Streptococcal strains. These proteins allow the pathogenic bacterium to evade the host immune response by coating the invading bacteria with host antibodies, thereby contributing significantly to the pathogenicity of these bacteria.

After carrying out the simulation, we will proceed analyzying its outcome illustrating several of the concepts we saw in the lecture.




## 2) Running This Notebook

We will need to install several packages:

*   **[GROMACS:](https://www.gromacs.org/)** MD engine. GROMACS also provides a set of `gmx` tools to analyze the trajectories. We will use those.
*   **[PyMol](https://pymol.org/2/):** for the visualization of the MD trajectories. **Please install this program in your computer.**    


In [ ]:
!sudo apt-get install gromacs
# pymol: install it in your laptop


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-mathjax gromacs-data libfftw3-double3 libfftw3-single3 libgromacs6
  libjs-mathjax sse4.2-support
Suggested packages:
  pymol libfftw3-bin libfftw3-dev fonts-mathjax-extras fonts-stix
  libjs-mathjax-doc
The following NEW packages will be installed:
  fonts-mathjax gromacs gromacs-data libfftw3-double3 libfftw3-single3
  libgromacs6 libjs-mathjax sse4.2-support
0 upgraded, 8 newly installed, 0 to remove and 3 not upgraded.
Need to get 58.3 MB of archives.
After this operation, 310 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 sse4.2-support amd64 6 [8,572 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-mathjax all 2.7.9+dfsg-1 [2,208 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfftw3-double3 amd64 3.3.8-2ubuntu8 [770 kB]
Get:4 http://archive

### Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## A)Preparation

<BR CLEAR=”left” />
<img align=right src="https://drive.google.com/uc?export=view&id=1PpM_OAUoVWWLqxhzi6ZUqiE8BE_8cpam" width=400>

The simulations will be carried out with the [GROMACS](https://www.gromacs.org/) simulation package. On the GROMACS homepage you can find both the software and documentation (online reference and manual). To run a simulation, three input files are usually required:

1.   **a structure** file, containing the initial atomic coordinates of the system to be simulated (**pdb or gro format**)
2.   **a "molecular topology"**, containing the atomic simulation parameters (force-field) and a description of the bonds, bond angles, etc (if any) of the simulation system (**top format**)
3.   **the simulation parameters**: type of simulation, number of steps, simulation temperature, etc. (**mdp format**)

We will now follow a standard protocol to run a typical MD simulation of a protein in a box of water in GROMACS. The individual steps are summarized in the [flowchart](https://mptg-cbp.github.io/teaching/tutorials/protein/flow.png) shown at the right.


**Note that** we can run linux terminal commands within the notebook cells. To achieve this, the command is preceded by **!**. That was actually the way we managed to install programs (e.g. using `pip`). GROMACS `gmx` run in that way, i.e. in a linux terminal. To test this you can retrieve the help menu:


In [ ]:
!gmx -h

### Initial conformation
Before a simulation can be started, an initial structure of the protein is required. Fortunately, the structure of the B1 domain of protein G has been solved experimentally, both by x-ray crystallography and NMR. Experimentally solved protein structures are collected and distributed by the [Protein Data Bank (PDB)](https://www.rcsb.org/). Please open this link in a new browser window and enter **"protein G" "B1"** in the search field. Several entries in the PDB should match this query. We will choose the x-ray structure with the highest resolution (**entry 1PGB**) for this study.

To download the structure, click on the link `1PGB`, and then, under `Download Files`, click the secondary button on `Legacy PDB format` and then select `Copy link`.

We download the structure with the command `wget`.


In [ ]:
!wget https://files.rcsb.org/download/1PGB.pdb # this is the link that was copied from the pdb.
# This file is stored in a local (and temporal) memory of google colab -check the file icon in the panel on the left.

--2026-07-11 13:08:42--  https://files.rcsb.org/download/1PGB.pdb
Resolving files.rcsb.org (files.rcsb.org)... 18.64.236.125, 18.64.236.33, 18.64.236.48, ...
Connecting to files.rcsb.org (files.rcsb.org)|18.64.236.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘1PGB.pdb’

1PGB.pdb                [ <=>                ]  57.19K  --.-KB/s    in 0.05s   

2026-07-11 13:08:43 (1.21 MB/s) - ‘1PGB.pdb’ saved [58563]



 To have a look at the contents of the downloaded PDB file

In [ ]:
with open('1PGB.pdb') as file:
  line=file.read()
  print(line)

HEADER    IMMUNOGLOBULIN BINDING PROTEIN          23-NOV-93   1PGB              
TITLE     TWO CRYSTAL STRUCTURES OF THE B1 IMMUNOGLOBULIN-BINDING DOMAIN OF     
TITLE    2 STREPTOCCOCAL PROTEIN G AND COMPARISON WITH NMR                      
COMPND    MOL_ID: 1;                                                            
COMPND   2 MOLECULE: PROTEIN G;                                                 
COMPND   3 CHAIN: A;                                                            
COMPND   4 ENGINEERED: YES                                                      
SOURCE    MOL_ID: 1;                                                            
SOURCE   2 ORGANISM_SCIENTIFIC: STREPTOCOCCUS SP. GX7805;                       
SOURCE   3 ORGANISM_TAXID: 1325                                                 
KEYWDS    IMMUNOGLOBULIN BINDING PROTEIN                                        
EXPDTA    X-RAY DIFFRACTION                                                     
AUTHOR    T.GALLAGHER,P.ALEX

In [ ]:
!more 1PGB.pdb


HEADER    IMMUNOGLOBULIN BINDING PROTEIN          23-NOV-93   1PGB              
TITLE     TWO CRYSTAL STRUCTURES OF THE B1 IMMUNOGLOBULIN-BINDING DOMAIN OF     
TITLE    2 STREPTOCCOCAL PROTEIN G AND COMPARISON WITH NMR                      
COMPND    MOL_ID: 1;                                                            
COMPND   2 MOLECULE: PROTEIN G;                                                 
COMPND   3 CHAIN: A;                                                            
COMPND   4 ENGINEERED: YES                                                      
SOURCE    MOL_ID: 1;                                                            
SOURCE   2 ORGANISM_SCIENTIFIC: STREPTOCOCCUS SP. GX7805;                       
SOURCE   3 ORGANISM_TAXID: 1325                                                 
KEYWDS    IMMUNOGLOBULIN BINDING PROTEIN                                        
EXPDTA    X-RAY DIFFRACTION                                                     
AUTHOR    T.GALLAGHER,P.ALEX

The file starts with general information about the protein, about the structure, and about the experimental techniques used to determine the structure, as well as literature references where the structure is described in detail. The data file contains the atomic coordinates of our protein structure with one line per atom (lines starting with `ATOM` and `HETATM`).

#### PyMol visualization
<BR CLEAR=”left” />
<img align=right src="https://mptg-cbp.github.io/teaching/tutorials/protein/snap_1pgb.png" width=300>



Let us visualize the structure with `PyMol`.

*   Click on the `Files` Button at the left menu and download the file `1PGB.pdb` to the local hard disk.
*   Open `PyMoL` in your computer.

*   In the PyMoL `File` menu, click `open` and select the downloaded `1PGB.pdb` file.

*   PyMoL displays the cartoon representation together with non-bonded crystallographic water molecules. Under the "Show" and "Hide" menus ("S" and "H" at the top-right of the "PyMOL Viewer" window), also try other representations such as "wire", "sticks", "spheres", and "surface". Try different views by moving the mouse over the molecule viewer with the left mouse button pressed, and zoom with the right mouse button pressed.
*   Exit pymol and continue in Google colab.

<font color=red size=+1> **QUESTION:** </font> Why do we start our MD simulations from the experimental determined 3D structure? Isn't it enough to know the proteins amino acid sequence?

##B) Setup

### add hydrogen atoms and topology generation

We will now prepare the protein structure to be simulated in GROMACS. Although we now have a starting structure for our protein, one might have noticed that hydrogen atoms (which would appear white) are still missing from the structure. This is because hydrogen atoms contain too few electrons to be observed by x-ray crystallography at moderate resolutions. Also, GROMACS requires a molecular description (or topology) of the molecules to be simulated before we can start, containing information on e.g. which atoms are covalently bonded and other physical information. Both the generation of hydrogen atoms and writing of the topology can be done with the GROMACS program `pdb2gmx`.

When prompted for the `force-field` and `water model` to be used, choose the number corresponding to the `OPLS-AA/L` all-atom force field and to the `SPC` water model.



In [ ]:
!gmx pdb2gmx -f 1PGB.pdb -o conf.pdb

             :-) GROMACS - gmx pdb2gmx, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

 View the resulting structure conf.pdb with `PyMol`:

* Download conf.pdb to your local hard disk

* Open PyMoL and load conf.pdb as done above.

* Show the protein in licorice->stick representation (hydrogens will be appear now in white).

**The topology file** written by pdb2gmx is called `"topol.top"`. Have a look at the contents of the file:


```
Double click on the `topol.top` file listed in the `"Files"` folder on the link menu.
```

You will see a list of:
*   all the atoms (with masses, charges)
*   bonds (covalent bonds connecting the atoms)
*   angles
*   dihedral angles
*   etc.

Near the very end of the topology (in the `"[molecules]"` section) there is a summary of the simulation system including:
*   the protein
*   24 crystallographic water molecules.

The topology file thus contains all the physical information about all interactions between the atoms of the protein (bonds, angles, torsion angles, Lennard-Jones interactions and electrostatic interactions).



### Solvate the protein
The next step in setting up the simulation system is to solvate the protein in a water box, to mimick a physiological environment. For that, we first need to define a simulation box. In this case we will generate a rectangular box with the box-edges at least 7 Angstroms apart from the protein surface:

In [ ]:
 ! gmx editconf -f conf.pdb -o box.pdb -d 0.7

             :-) GROMACS - gmx editconf, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff           

 Note that GROMACS uses units of nanometers. View the result with 'PyMoL':

 * Download `box.pdb` to your local hard disk

* Open PyMoL and load box.pdb.

* show the protein as lines, by typing in the PyMoL prompt (white area):

   `show lines`

* show the simulation box, by typing in the PyMoL prompt (white area):
   `show cell`
   
   you may need to zoom out to see the box

 Now we fill the simulation box with SPC water using:

In [ ]:
! gmx solvate -cp box.pdb -cs spc216 -o water.pdb -p topol.top


             :-) GROMACS - gmx solvate, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

Again, view the output (water.pdb) with `PyMoL`

Now the simulation system is almost ready.

### Energy minimisation
Before we can start the dynamics, we must perform an **energy minimisation**, to alleviate any bad contacts (atoms overlapping such that a significant repulsion would result, causing numerical problems in the simulation) that might be present in the system. For this we need a parameter file, specifying which type of minimisation should be carried out, the number of steps, etc. For your convenience this file, called `"em.mdp"`, has already been prepared and can be downloaded as:

In [ ]:
! wget https://mptg-cbp.github.io/teaching/tutorials/protein/em.mdp

--2026-06-22 11:25:40--  https://mptg-cbp.github.io/teaching/tutorials/protein/em.mdp
Resolving mptg-cbp.github.io (mptg-cbp.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to mptg-cbp.github.io (mptg-cbp.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 489 [application/octet-stream]
Saving to: ‘em.mdp’

em.mdp              100%[===================>]     489  --.-KB/s    in 0s      

2026-06-22 11:25:40 (24.7 MB/s) - ‘em.mdp’ saved [489/489]



View the file double-clicking on it to see its contents.

We use the GROMACS preprocessor (`grompp`) to prepare our energy minimisation.

`grompp` collects all the information from `em.mdp`, the coordinates from `water.pdb` and the topology from `topol.top`, checks if the contents are consistent and writes a unified output file: `em.tpr`:

In [ ]:
! gmx grompp -f em.mdp -c water.pdb -p topol.top -o em.tpr -maxwarn 2

              :-) GROMACS - gmx grompp, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

Finally `em.tpr` is used to carry out the minimisation:

In [ ]:
!gmx mdrun -v -s em.tpr -c em.pdb

              :-) GROMACS - gmx mdrun, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            T

The output shows that already the initial energy was rather low, so in this case there were hardly any bad contacts.

Having a look at "em.pdb" shows that the structure hardly changed during minimisation.

### Charge neutralization

The careful user may have noticed that grompp gave a warning NOTE:

```
System has non-zero total charge: -4.
```


Before we continue with the dynamics, we should neutralise this net charge of the simulation system. This is to prevent artefacts that would arise as a side effect caused by the periodic boundary conditions used in the simulation. A net charge would result in an electrostatic repulsion between neighbouring periodic images. Therefore, 4 sodium ions will be added to the system:



In [ ]:
!gmx genion -s em.tpr -o ions.pdb -np 4 -pname NA -p topol.top

# when prompted: select the number corresponding to the water group (SOL) from
# which 4 water molecules will be replaced by sodium ions.

              :-) GROMACS - gmx genion, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

 The output (`ions.pdb`) can be checked with `PyMoL`, to see the ions:

 * Download `ions.pdb` to your local hard disk and open it with PyMoL.

* show the ions as purple spheres, by typing in the PyMoL prompt (white area):

   `color purple, resn NA`

   `show spheres, resn NA`



 Have a look at the end of the topoloy file `topol.top`. You will see that 4 water molecules were replaced by sodium ions.

Just to be on the safe side, we repeat the energy minimisation, now with the ions included (remember to (re)run `grompp` to create a new run input file whenever changes to the topology, or coordinates have been made):

In [ ]:
!gmx grompp -f em.mdp -c ions.pdb -p topol.top -o em.tpr -maxwarn 2
!gmx mdrun -v -s em.tpr -c em.pdb

              :-) GROMACS - gmx grompp, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

##C) Simulation
Now we have all that is required to start the dynamics. Again, a simulation parameter file has been prepared for the simulation:

In [ ]:
!wget https://mptg-cbp.github.io/teaching/tutorials/protein/md.mdp

--2026-06-22 11:33:39--  https://mptg-cbp.github.io/teaching/tutorials/protein/md.mdp
Resolving mptg-cbp.github.io (mptg-cbp.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to mptg-cbp.github.io (mptg-cbp.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7994 (7.8K) [application/octet-stream]
Saving to: ‘md.mdp’

md.mdp              100%[===================>]   7.81K  --.-KB/s    in 0s      

2026-06-22 11:33:39 (62.0 MB/s) - ‘md.mdp’ saved [7994/7994]



Please browse through the file `"md.mdp"` (right click on it at the left "Files" folder) to get an idea of the simulation parameters.

The GROMACS online manual describes all parameters in detail. Please don't worry in this stage about all individual parameters, we've chosen common values typical for protein simulations.

Again, we use the GROMACS preprocessor `grompp` to prepare the simulation:

In [ ]:
!gmx grompp -f md.mdp -c em.pdb -p topol.top -o md.tpr -maxwarn 2

              :-) GROMACS - gmx grompp, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

 and start the simulation!

In [ ]:
!gmx mdrun -v -s md.tpr -c md.pdb

              :-) GROMACS - gmx mdrun, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            T

 The simulation is running now, and depending on the speed and load of the computer, the simulation will run for a number of minutes.


**QUESTION:** How do the parts of energy minimization and MD simulation differ with reference to the energy landscape $V\left(\vec{r1},\vec{r2},...,\vec{r_N}\right)$?

##D) Analysis of a GROMACS simulation

The simulation is running now (or finished) and we can start analysing the results. Let us first see which kind of files have been written by the simulation (`mdrun`):

In [ ]:
!ls -lhrt

 We see the following files:
*   traj_comp.xtc - the trajectory (coordinates) to be used for analyses
*   traj.trr - coordinates, velocities and forces written at less frequency.
*    state.cpt - a state of the trajectory to be used for a restart in case of a crash
*    ener.edr - energies
*    md.log - a LOG file of mdrun
*    md.pdb - the final coordinates of the simulation

### 1) Exploratory data analysis

<BR CLEAR=”left” />
<img align=right src="https://drive.google.com/uc?export=view&id=14BIqfDSoF96r9lXMMrlgmk0xZozjzTrE" width=400>


The first analysis step during or after a simulation is usually a visual inspection of the trajectory. For this we will use `PyMol`.

* Download to your local hard disk the following files:
 - `md.pdb`: final coordinates
 - `traj_comp.xtc`: trajectory

* Open PyMoL
* **Load the final conformation:**
  
  `File` menu $\to$ `open` $\to$ select the downloaded `md.pdb` file.

* **Load the trajectory,** by typing in the PyMoL prompt (white area):

   `load ~/PATH/traj_comp.xtc,  md`

   `PATH` is the directory where you saved the trajectory. For instance `Downloads`.

If everything worked well, 102 states should have been read.



Play the animation by pressing the play button.
We can see that the protein and its surroundings undergo thermal
fluctuations, but overall, the protein structure is rather stable, as
would be expected on such timescales. To change the view orientation, move the mouse over the molecule, with either the left or right mouse buttons pressed.

### 2) Protein dynamics
As mentioned in the lecture, if we are interested only on the protein dynamics, in the analysis we can omit the solvent. If you wish to visualize only the protein (in cartoon and stick representation), on the pymol prompt, type:
<p>
</p><div class="code">
<pre>
hide all
show cartoon
show sticks , poly
</pre>
</div>
<p>
Play the animation by pressing the play button.


### 3) Removal of rigid body translation and rotation degrees of freedom

We want the properties we calculate to be invariant under rotation and translation. To this end, we remove the rigid body rotation and translation of the protein. This is done automatically by most of the GROMACS analysis tools. However, we will do this step explicitly for visualization.

Execute the following `trjconv` GROMACS command:

when prompted for groups select:
* "Backbone" for the least squares fit group
* "Protein" for the output group


In [ ]:
!gmx trjconv -f traj_comp.xtc -s md.tpr -fit rot+trans -o protein_fit.xtc

`protein_fit.xtc` contains the trajectory of the protein only (no solvent). We will also need a reference conformation for the protein without the solvent:

again select `"Backbone"` and `"Protein"` when prompted.

In [ ]:
!gmx trjconv -f traj_comp.xtc -s md.tpr -fit rot+trans -o protein_0ns.pdb -dump 0

Download `protein_0ns.pdb` and `protein_fit.xtc` and visualize the fitted trajectory with PyMoL. The trajectory looks very similar to the original trajectory, because the protein did not deviate significantly from its initial position in so short time scale. This may not be the case when we simulate for longer times.

### 4) RMSD
For a more quantitative analysis
on the protein fluctuations, we can view how fast and how far the
protein deviates from the starting (experimental) structure:<br> <p>

When prompted for groups to be analysed, type "1" two times (meaning `Protein` both for the least square fit and RMSD calculation.

In [ ]:
!gmx rms -s md.tpr -f traj_comp.xtc


`gmx rms` has written a file called `"rmsd.xvg"`, which can be viewed with:

In [ ]:
data = np.loadtxt('rmsd.xvg', comments=[ '#', '@'], unpack=True)
plt.plot(data[0], data[1])
plt.xlabel("Time (ps)")
plt.ylabel("RMSD (nm)")

We see the Root Mean Square Deviation (RMSD) from the starting
structure, averaged over all protein atoms, as a function of time.  

<font color=red size=+1> **QUESTION:** </font> Why is there an intial rise in the rmsd?

### 5) RMSF
If we now wish to see if the fluctuations are equally distributed over the
protein, or if some residues are more flexible than others, we can
type (selecting group `"4" Backbone`):

In [ ]:
!gmx rmsf -s md.tpr -f traj_comp.xtc -oq -res


The result can be viewed with:

In [ ]:
data = np.loadtxt('rmsf.xvg', comments=[ '#', '@'], unpack=True)
plt.plot(data[0], data[1])
plt.xlabel("Residue")
plt.ylabel("RMS fluctuation (nm)")

We can see that mainly four regions in the protein show a large
flexibility: around residues 1, 11, 21 and 38. To see where these
residues are located in the protein, we can use `PyMoL`:


* Download `bfac.pdb` to your local hard disk and open it with `PyMoL`.

* show the protein as spheres, by typing in the PyMoL prompt (white area):

   `show spheres`

* show the `RMSF` as color:
   
   Color menu `C` at the right-hand side$\to$ `spectrum` $\to$`b-factors`
   
   

The protein backbone is now
shown with the flexibility encoded in the colour. The red (and orange) regions are relatively flexible and the white regions are
relatively rigid. It can be seen that the alpha-helix and beta-sheet
are relatively stable, whereas the loops are more flexible.

<font color=red size=+1> **QUESTION:** </font> Which is the most flexible residue?


###6) POTENTIAL ENERGY
The simulation not only yields information on the structural
properties of the simulation, but also on the energetics. The energies written by mdrun can be analysed:


In [ ]:
!gmx energy -f ener.edr

Select `"Potential"` and end your selection by pressing enter twice, View the result with:

In [ ]:
data = np.loadtxt('energy.xvg', comments=[ '#', '@'], unpack=True)
plt.plot(data[0], data[1])
plt.xlabel("Time (ps)")
plt.ylabel("Potential Energy (kJ/mol)")
#plt.ylabel("Temperature (K)")

As can be seen, the total potential energy initially rises rapidly after
which it relaxes again. <br> <p>
<font color=red size=+1> **QUESTION**: </font>Can you think of an explanation for
this behaviour?<br> <p>
Please repeat the energy analysis for a number of different energy
terms to obtain an impression of their behaviour.<br> <p>
<font color=red size=+1> **QUESTION**: </font> Do you think the length of our
simulation is sufficient to provide a faithful picture of the protein's
conformations at equillibrium. <p>

###7) RADIUS OF GYRATION
The next thing to analyse is the change in the overall size (or gyration radius) of the protein:

In [ ]:
!gmx gyrate -s md.tpr -f traj_comp.xtc

(again, select group "1" for the protein).

In [ ]:
data = np.loadtxt('gyrate.xvg', comments=[ '#', '@'], unpack=True)
plt.plot(data[0], data[1])
plt.xlabel("Time (ps)")
plt.ylabel("r$_g$ (nm)")

<font color=red size=+1> **QUESTION:** </font>
What can you say about the size and overall shape of the protein based on the time-trace of the radius of gyration (e.g. fluctuations and drift)?  


### 8) SOLVENT ACCESSIBLE SURFACE AREA (SASA)
Another important check
concerns the behaviour of the protein surface:


In [ ]:
!gmx sasa -s md.tpr -f traj_comp.xtc -surface 'Protein' -output 'Protein'

Now view the total solvent accessible surface area with:

In [ ]:
data = np.loadtxt('area.xvg', comments=[ '#', '@'], unpack=True)
plt.plot(data[0], data[1])
plt.xlabel("Time (ps)")
plt.ylabel("Area (nm$^2$)");

Let us now split the area into hydrophobic and hydrophilic contributions:

In [ ]:
!gmx sasa -s md.tpr -f traj_comp.xtc  -surface 'Protein' -output '"Hydrophobic" group "Protein" and charge {-0.2 to 0.2}; "Hydrophilic" group "Protein" and not charge {-0.2 to 0.2}; "Total" group "Protein"'


and visualize again the output with:

In [ ]:
data = np.loadtxt('area.xvg', comments=[ '#', '@'], unpack=True)
time=data[0]
area_total=data[1]
area_hydrophobic=data[2]
area_hydrophilic=data[3]

plt.plot(time, area_hydrophobic)
plt.plot(time, area_hydrophilic)

plt.legend(['Hydrophobic', 'Hydrophilic'])

plt.xlabel("Time (ps)")
plt.ylabel("Area (nm$^2$)");

<font color=red size=+1> **QUESTION:** </font> Is the total
(solvent-accessible) surface constant? Are there any hydrophobic groups exposed to the solvent during the simulation? (hint: check the type of residues exposed to the solvent with `PyMoL`).


##7)References:
* Brini, Emiliano, Carlos Simmerling, and Ken Dill. **Protein Storytelling Through Physics.** [Science 370:6520 (2020)](https://doi.org/10.1126/science.aaz3041).

* Ron O. Dror, Robert M. Dirks, J.P. Grossman, Huafeng Xu, and David E. Shaw. **Biomolecular Simulation: A Computational Microscope for Molecular Biology.** [Annual Review of Biophysics. 41:429-452 (2012).](https://doi.org/10.1146/annurev-biophys-042910-155245)


* M. Karplus and A. McCammon.  **Molecular Dynamics simulations of
biomolecules.** [Nature structural biology 9: 646-652 (2002).](http://www.nature.com/nsmb/journal/v9/n9/full/nsb0902-646.html)

* D.C. Rapaport. **The Art of Molecular Dynamics Simulations - 2nd edn**.Cambridge University Press. (2004).